# Week 1, Your First Investigation  (fully worked)

**What you'll do today.** Load a real dataset, ask it a question, and make your first chart,
a complete investigation end to end, before any theory. Along the way you'll set up the one
ritual that carries the whole course: your Drive project folder.

In [ ]:
# If an import fails: re-run the install cell above; if it persists see ../kits/common-errors-cheatsheet.md
# (standalone copy: https://github.com/lucianli123/culture-as-data-2026/blob/main/kits/common-errors-cheatsheet.md)
# --- Make your work survive a Colab reset -------------------------------------
# Colab wipes the runtime when it disconnects or idles out. Mount your Google Drive
# and keep everything in ONE project folder, so your corpus, embeddings, labels, and
# charts are still there next week. (Outside Colab - e.g. the offline test harness -
# this falls back to a local folder so the notebook still runs.)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder:", PROJECT_DIR)

## No installs today

Everything Week 1 needs ships with Colab already. (Later weeks have an install cell; today
the only setup is the Drive mount above, and, in class, putting your free Gemini key into
Colab Secrets for Week 7.)

In [ ]:
import io, os, re
import pandas as pd
import matplotlib.pyplot as plt
SMOKE = os.environ.get("CULTURE_AS_DATA_SMOKE") == "1"
print("SMOKE mode (tiny, offline):", SMOKE)

## Load real culture: 30 years of NYT wedding announcements

A CSV assembled by The Upshot: every NYT wedding announcement 1985–2014, whether the bride
kept or took a name, and the couple's ages. One `pd.read_csv(url)` and it's a table.

In [ ]:
SAMPLE = """url,name_status,bride_age,groom_age
http://www.nytimes.com/1986/06/01/style/a.html,taking,24,26
http://www.nytimes.com/1986/09/07/style/b.html,taking,23,25
http://www.nytimes.com/1991/06/16/style/c.html,taking,26,28
http://www.nytimes.com/1991/08/04/style/d.html,keeping,27,27
http://www.nytimes.com/1996/05/12/style/e.html,taking,27,29
http://www.nytimes.com/1996/10/20/style/f.html,keeping,29,30
http://www.nytimes.com/2001/06/03/style/g.html,keeping,29,31
http://www.nytimes.com/2001/09/09/style/h.html,taking,28,30
http://www.nytimes.com/2006/06/11/style/i.html,keeping,30,32
http://www.nytimes.com/2006/11/05/style/j.html,keeping,31,31
http://www.nytimes.com/2011/06/12/style/k.html,keeping,31,33
http://www.nytimes.com/2011/10/16/style/l.html,taking,30,32
http://www.nytimes.com/2014/06/08/style/m.html,keeping,32,33
http://www.nytimes.com/2014/09/14/style/n.html,keeping,33,35
"""

URL = "https://raw.githubusercontent.com/TheUpshot/nyt_weddings/master/nyt_wedding_announcements.csv"
if SMOKE:
    weddings = pd.read_csv(io.StringIO(SAMPLE))
    print("offline sample:", len(weddings), "announcements")
else:
    try:
        weddings = pd.read_csv(URL)
        print("loaded from the web:", len(weddings), "announcements")
    except Exception as e:
        weddings = pd.read_csv(io.StringIO(SAMPLE))
        print("no network, using the small built-in sample:", e)
weddings.head()

## Ask it a question, and answer with a chart

Each announcement records whether the bride is *keeping* or *taking* a name, and the article
URL contains the date. Question: **did name-keeping become more common over these 30 years?**
The answer is three lines of code and one picture.

In [ ]:
weddings["year"] = weddings["url"].str.extract(r"nytimes\.com/(\d{4})/").astype(float)
by_year = (weddings[weddings["name_status"].isin(["keeping", "taking"])]
           .groupby("year")["name_status"]
           .apply(lambda s: (s == "keeping").mean()))

plt.figure(figsize=(7, 4))
plt.plot(by_year.index, by_year.values * 100, marker="o")
plt.ylabel("% of brides keeping their name")
plt.xlabel("year of announcement")
plt.title("Name-keeping in NYT wedding announcements")
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, "week01_first_chart.png"), dpi=150)
plt.show()
print("chart saved to your project folder")

**Before you trust it, one question the whole course turns on:** *whose* weddings are these?
NYT announcements skew wealthy, East-Coast, and socially prominent. The chart is real; the
"America" it describes is a slice. Naming that isn't a gotcha, it's the skill.

## The unblocking drill (you will need this)

The AI will sometimes hand you code that errors. The move, printed on the cheat sheet
(`kits/common-errors-cheatsheet.md`), is: **read the last line of the traceback, paste it back
to the AI with "this errored, fix it and tell me what went wrong," try twice, then ask a
human.** Here's the drill on a bug you'll actually meet. This line looks right and isn't:

```python
by_year = weddings.groupby("Year")["name_status"].count()   # KeyError: 'Year'
```

The last line of that traceback says `KeyError: 'Year'`, the column is called `year`
(lowercase), and `weddings.columns` would have told you. Run the fixed version:

In [ ]:
# The fix: column names are exact. When a KeyError names a column, print the real ones.
print("the actual columns:", list(weddings.columns))
by_year_count = weddings.groupby("year")["name_status"].count()
by_year_count.head()

## Lab 2: count an image corpus, pixel by pixel

Pictures are data the same way text is. A picture is a grid of numbers, each pixel a
red/green/blue triple, so "how dark is this image?" is a counting question. In class this
corpus is ~200 Met Museum thumbnails (or a stack of album covers) prepared in your Drive
folder; the cell below carries a small built-in stand-in corpus so it runs anywhere.

In [ ]:
import numpy as np
from PIL import Image

# The stand-in image corpus: solid-color "covers." In class, replace this with the
# prepared Met-thumbnail folder in your Drive (the counting code is identical).
def swatch(rgb, size=64):
    arr = np.zeros((size, size, 3), dtype=np.uint8)
    arr[:, :] = rgb
    return Image.fromarray(arr)

corpus_images = {
    "midnight":  swatch((18, 18, 42)),
    "storm":     swatch((70, 80, 95)),
    "forest":    swatch((30, 90, 50)),
    "terracotta":swatch((150, 80, 60)),
    "gold":      swatch((205, 160, 60)),
    "sunburst":  swatch((240, 190, 60)),
    "cream":     swatch((235, 225, 205)),
}

def avg_brightness(img):
    a = np.asarray(img).astype(float)
    return float((0.299*a[...,0] + 0.587*a[...,1] + 0.114*a[...,2]).mean())  # perceived luminance

def avg_color(img):
    return tuple(np.asarray(img).reshape(-1, 3).mean(axis=0).astype(int))

ranked = sorted(corpus_images, key=lambda k: avg_brightness(corpus_images[k]))
print("darkest -> brightest:", ranked)
for k in ranked:
    print(f"  {k:<11} brightness {avg_brightness(corpus_images[k]):6.1f}   avg color {avg_color(corpus_images[k])}")

plt.figure(figsize=(7, 3.5))
plt.bar(ranked, [avg_brightness(corpus_images[k]) for k in ranked],
        color=[np.asarray(corpus_images[k]).reshape(-1,3).mean(0)/255 for k in ranked])
plt.ylabel("average brightness"); plt.title("An image corpus, counted at the pixel level")
plt.tight_layout(); plt.show()

# The choice to notice: we counted BRIGHTNESS. Counting average hue, or saturation, or
# edge density would rank this corpus differently. What you choose to count of a picture
# is a decision, exactly like the choices you'll make about words next week.

## You did a complete investigation

Loaded real culture, asked a question, answered it with a chart, questioned the chart's
scope, recovered from an error like a pro, and counted a stack of images at the pixel level, pictures and text as the same kind of data from day one. Everything
else in this course is this loop with sharper tools, and it's all saved in your Drive
project folder, which will still be there next week.

**Sketch (homework):** one question from your own life you could answer with text or image
data; three sentences. No code.